# Exercício Complementar — LSTM (SLR)

Notebook formatado para **Google Colab** conforme enunciado:
- texto em `title-abstract`
- rótulo em `label`
- Embedding 300
- 2 camadas LSTM (300 neurônios cada)
- Dense 300 + saída com 1 neurônio
- máximo de 200 tokens
- treino por 10 épocas
- avaliação com acurácia, precisão, recall e F-score por classe


In [ ]:
# (Opcional) Dependências no Colab
!pip -q install pandas scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


In [ ]:
# Configurações fixas do enunciado
DATA_URL = 'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/slr.csv'
EPOCHS = 10
MAX_TOKENS = 200
MAX_WORDS = 50000
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


In [ ]:
# Carga e validação

df = pd.read_csv(DATA_URL)
required = {'title-abstract', 'label'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Colunas ausentes no dataset: {missing}')

X = df['title-abstract'].fillna('').astype(str)
y_raw = df['label'].astype(str)

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

if len(encoder.classes_) != 2:
    raise ValueError(
        'Este notebook espera classificação binária (saída com 1 neurônio). '
        f'Classes encontradas: {list(encoder.classes_)}'
    )

print('Classes:', list(encoder.classes_))


In [ ]:
# Split + tokenização + padding
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_TOKENS, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_TOKENS, padding='post', truncating='post')

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
print('Vocab size:', vocab_size)


In [ ]:
# Modelo LSTM conforme enunciado
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=300, input_length=MAX_TOKENS),
    LSTM(300, return_sequences=True),
    LSTM(300),
    Dense(300, activation='relu'),
    Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Treino (10 épocas)
model.fit(X_train_pad, y_train, epochs=EPOCHS, batch_size=32, verbose=1)


In [ ]:
# Teste e métricas

y_proba = model.predict(X_test_pad, verbose=0).reshape(-1)
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
precision, recall, fscore, support = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1], zero_division=0
)

print(f'Acurácia geral: {acc:.4f}')
print('\nMétricas por classe:')
for idx, class_name in enumerate(encoder.classes_):
    print(
        f"- Classe '{class_name}': "
        f"Precision={precision[idx]:.4f}, "
        f"Recall={recall[idx]:.4f}, "
        f"F-Score={fscore[idx]:.4f}, "
        f"Support={support[idx]}"
    )
